# FIT5217 Assignment 2 - Task 3: Menu Designer (RAG + LLM-as-Judge)

**Name:** Jili Chen  
**Student ID:** 35423757

**AI Usage Declaration:** Option 3 - I used generative AI tools for coding support, debugging, understanding the RAG pipeline, and drafting analysis. All outputs were reviewed, validated, and adapted by me.

## Table of Contents

1. T3.1 Knowledge Base Construction
2. T3.2 RAG Pipeline for Menu Generation
3. T3.3 LLM-as-Judge Evaluation
4. T3.4 Analysis and Reflection


In [1]:
import sys
import os
import json
import pickle
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.append(str(ROOT))
sys.path.append("..")

from src.utils import set_seed

set_seed(42)

is_colab = "google.colab" in sys.modules
print(f"Environment: {'Colab' if is_colab else 'Local'}")
print(f"Project root: {ROOT}")
print("Random seed fixed at 42.")
print("This notebook does not load API keys, call any LLM API, rebuild the KB, or run retrieval.")


Environment: Local
Project root: /Users/nili/Desktop/fit5217_a2
Random seed fixed at 42.
This notebook does not load API keys, call any LLM API, rebuild the KB, or run retrieval.


## T3.1 Knowledge Base Construction

The knowledge base was built over the full train, development, and test splits because Task 3 is a retrieval and menu-design task rather than a supervised model-training task. Unlike Tasks 1 and 2, using the test recipes as retrievable documents is allowed for the RAG component. I selected BM25 because it is fast to index, CPU friendly, deterministic, and well matched to cooking queries where ingredient and dish-name keywords matter. Each document concatenates the recipe title, ingredients, and recipe instructions, giving the retriever access to both short semantic labels and detailed cooking evidence. Two lightweight normalization steps were added: stopword filtering and plural normalization. These reduce the influence of generic words such as “give”, “some”, and “recipes”, while mapping variants such as “recipes” and “recipe” closer together. This is useful because menu queries often contain broad request language that can otherwise dominate the ranking.

In [2]:
kb_stats = pd.DataFrame([
    {"Metric": "Documents", "Value": "165,045"},
    {"Metric": "Indexing time", "Value": "5.66 sec"},
    {"Metric": "Avg doc length (tokens)", "Value": "54.13"},
    {"Metric": "Retrieval latency", "Value": "37.39 ms"},
    {"Metric": "Retrieval method", "Value": "BM25"},
    {"Metric": "Index size", "Value": "258 MB"},
])
display(kb_stats)
print("KB statistics are displayed from the recorded build output; kb.pkl is not loaded in this notebook.")


,Metric,Value
0,Documents,"165,045"
1,Indexing time,5.66 sec
2,Avg doc length (tokens),54.13
3,Retrieval latency,37.39 ms
4,Retrieval method,BM25
5,Index size,258 MB


KB statistics are displayed from the recorded build output; kb.pkl is not loaded in this notebook.


In [3]:
def load_menu_demo(demo_id):
    path = ROOT / "outputs" / "menus" / f"demo_{demo_id}.json"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

menus = {f"demo_{i}": load_menu_demo(i) for i in [1, 2, 3]}

retrieval_rows = []
for demo_name, data in menus.items():
    for rank, recipe in enumerate(data["retrieved_recipes"][:5], start=1):
        retrieval_rows.append({
            "Demo": demo_name,
            "Rank": rank,
            "Title": recipe["title"],
            "Key Ingredients": "; ".join(recipe["ingredients"][:5]),
            "Score": round(recipe.get("score", 0.0), 3),
        })

retrieval_df = pd.DataFrame(retrieval_rows)
display(retrieval_df)
print("Displayed top-5 retrieved recipes per demo from saved RAG outputs only.")


,Demo,Rank,Title,Key Ingredients,Score
0,demo_1,1,Surprise Fruit Salad,1 can Del Monte tropical fruit; 2 bananas; 1 o...,21.393
1,demo_1,2,Tortellini Soup,3 cans chicken broth; 1 (12 oz.) pkg. tortelli...,19.863
2,demo_1,3,Marinated Chicken Breasts,"1-inch ginger root, chopped fine; 1/2 c. teriy...",19.661
3,demo_1,4,Chicken-Cheese Soup,1 container chicken noodle soup starter; 1 1/2...,19.190
4,demo_1,5,Old-Fashioned Chicken Main-Dish Soup,1 can chicken noodle soup; 1 can chicken with ...,19.097
5,demo_2,1,Cheesy Pizza Dinner,1 (14 oz.) pkg. Kraft Deluxe macaroni and chee...,23.822
6,demo_2,2,Candle Stick Salad(For Holiday Dinner Party),8 washed lettuce leaves *; 8 slices canned pin...,22.120
7,demo_2,3,Surprise Fruit Salad,1 can Del Monte tropical fruit; 2 bananas; 1 o...,21.393
8,demo_2,4,Pink Cloud Dessert,1 (12 oz.) Cool Whip; 1 small can crushed pine...,20.825
9,demo_2,5,Bachelor'S Banquet,1 T.V. Dinner; 1 glass white wine,20.744


Displayed top-5 retrieved recipes per demo from saved RAG outputs only.


The retrieval results show reasonable keyword grounding for the three menu levels. Demo 1 includes source recipes connected to the requested soup, chicken, and cake courses, which gives the generator enough evidence for a simple three-course menu. Demo 2 retrieves Italian-themed items such as Italian Skillet Pasta, Antipasto Salad, and Biscuit Tortoni, supporting both cuisine and course constraints. Demo 3 retrieves vegetarian and fruit-based recipes that can be adapted toward a low-fat menu. A known limitation is that BM25 is surface-level: it can miss conceptually relevant dishes when the exact words are absent, such as Italian dishes whose titles do not include “pasta” or “Italian”.

## T3.2 RAG Pipeline for Menu Generation

The menu designer uses a three-stage RAG pipeline: retrieval, prompt construction, and generation. The generator is Groq `llama-3.3-70b-versatile`; credentials are read from an environment variable in the script, with no hardcoded API key. For this submission notebook, the API is not called and saved outputs are loaded instead. A single top-k retrieval can under-cover multi-course constraints, so the implementation uses `top_k=12` plus auxiliary course-specific queries. For example, a request mentioning soup, chicken, and cake triggers additional probes for those course keywords, and the results are deduplicated into up to 24 source recipes. The prompt asks the LLM to return JSON only, with course labels, dish names, exact source recipe titles, and explicit constraint notes. This design keeps the generated menu auditable: every dish can be traced back to retrieved recipes, and every constraint must be justified in natural language rather than left implicit.

In [4]:
menu_prompt_template = '''SYSTEM:
You are a menu designer for a university NLP assignment.
Use only the retrieved recipe context as source material.
Return valid JSON only. Do not include markdown fences or prose outside JSON.
Every dish must cite at least one source recipe title exactly as shown in context.
Every course must explicitly explain how it satisfies the user's constraints.

USER:
User request:
{query}

Retrieved recipe context:
[1] Title: {title}
Ingredients: {ingredients}
Recipe excerpt: {recipe_excerpt}
BM25 score: {score}
...

Create a menu that satisfies the request. Return this exact JSON shape:
{
  "menu": [
    {
      "course": "Starter",
      "dish": "Dish name",
      "source_recipes": ["Exact source recipe title"],
      "constraint_notes": "Explain explicitly how this dish satisfies the user's course and dietary constraints."
    }
  ],
  "overall_notes": "Briefly explain how the full menu fits together."
}

Rules:
- Include exactly the courses requested by the user, normally 3 courses.
- Use course labels such as Starter, Main, Dessert.
- Each dish must cite at least one source recipe title from the retrieved context.
- If the user says avoid an ingredient, do not include it in the dish and mention the avoidance in constraint_notes.
- If context is imperfect, adapt conservatively but keep source_recipes grounded in retrieved titles.
'''

display(Markdown("```text\n" + menu_prompt_template + "\n```"))
print("Displayed the core menu-generation prompt template; no LLM call was made.")


```text
SYSTEM:
You are a menu designer for a university NLP assignment.
Use only the retrieved recipe context as source material.
Return valid JSON only. Do not include markdown fences or prose outside JSON.
Every dish must cite at least one source recipe title exactly as shown in context.
Every course must explicitly explain how it satisfies the user's constraints.

USER:
User request:
{query}

Retrieved recipe context:
[1] Title: {title}
Ingredients: {ingredients}
Recipe excerpt: {recipe_excerpt}
BM25 score: {score}
...

Create a menu that satisfies the request. Return this exact JSON shape:
{
  "menu": [
    {
      "course": "Starter",
      "dish": "Dish name",
      "source_recipes": ["Exact source recipe title"],
      "constraint_notes": "Explain explicitly how this dish satisfies the user's course and dietary constraints."
    }
  ],
  "overall_notes": "Briefly explain how the full menu fits together."
}

Rules:
- Include exactly the courses requested by the user, normally 3 courses.
- Use course labels such as Starter, Main, Dessert.
- Each dish must cite at least one source recipe title from the retrieved context.
- If the user says avoid an ingredient, do not include it in the dish and mention the avoidance in constraint_notes.
- If context is imperfect, adapt conservatively but keep source_recipes grounded in retrieved titles.

```

Displayed the core menu-generation prompt template; no LLM call was made.


In [5]:
def render_menu_demo(demo_name, title):
    data = menus[demo_name]
    lines = [f"### {title}", "", f"**Query:** {data['query']}", "", "**Top-5 retrieved recipes:**"]
    for i, recipe in enumerate(data["retrieved_recipes"][:5], start=1):
        lines.append(f"{i}. {recipe['title']}")
    lines.extend(["", "**Generated menu:**"])
    for item in data["menu"]:
        sources = ", ".join(item.get("source_recipes", []))
        lines.append(f"- **{item.get('course', '')}: {item.get('dish', '')}**")
        lines.append(f"  - Source recipes: {sources}")
        lines.append(f"  - Constraint notes: {item.get('constraint_notes', '')}")
    lines.extend(["", f"**Overall notes:** {data.get('overall_notes', '')}"])
    display(Markdown("\n".join(lines)))
    print(f"Loaded {demo_name}: {len(data['retrieved_recipes'])} retrieved recipes, {len(data['menu'])} menu courses.")

render_menu_demo("demo_1", "Demo 1: L1 - Simple")


### Demo 1: L1 - Simple

**Query:** Suggest a 3-course dinner menu: a soup starter, a chicken main dish, and a cake for dessert.

**Top-5 retrieved recipes:**
1. Surprise Fruit Salad
2. Tortellini Soup
3. Marinated Chicken Breasts
4. Chicken-Cheese Soup
5. Old-Fashioned Chicken Main-Dish Soup

**Generated menu:**
- **Starter: Tortellini Soup**
  - Source recipes: Tortellini Soup
  - Constraint notes: This dish satisfies the user's request for a soup starter. It is based on the Tortellini Soup recipe, which provides a clear and simple soup option.
- **Main: Marinated Chicken Breasts**
  - Source recipes: Marinated Chicken Breasts
  - Constraint notes: This dish satisfies the user's request for a chicken main dish. The Marinated Chicken Breasts recipe provides a flavorful and well-rounded main course option.
- **Dessert: Pineapple Delight Cake**
  - Source recipes: Pineapple Delight Cake
  - Constraint notes: This dish satisfies the user's request for a cake dessert. The Pineapple Delight Cake recipe provides a sweet and satisfying dessert option.

**Overall notes:** The full menu fits together by providing a balanced and varied three-course meal. The Tortellini Soup starter offers a light and savory beginning, followed by the Marinated Chicken Breasts main course, which provides a rich and flavorful centerpiece. Finally, the Pineapple Delight Cake dessert offers a sweet and indulgent conclusion to the meal.

Loaded demo_1: 22 retrieved recipes, 3 menu courses.


In [6]:
render_menu_demo("demo_2", "Demo 2: L2 - Intermediate Italian")


### Demo 2: L2 - Intermediate Italian

**Query:** Plan a 3-course Italian dinner (starter, pasta main, dessert) for a dinner party.

**Top-5 retrieved recipes:**
1. Cheesy Pizza Dinner
2. Candle Stick Salad(For Holiday Dinner Party)  
3. Surprise Fruit Salad
4. Pink Cloud Dessert
5. Bachelor'S Banquet

**Generated menu:**
- **Starter: Antipasto Salad**
  - Source recipes: Antipasto Salad, Antipasto Turkey Salad
  - Constraint notes: This dish satisfies the user's starter course constraint by providing a light and refreshing Italian-inspired salad. It includes a variety of ingredients such as olives, peppers, and salami, which are commonly found in Italian cuisine.
- **Main: Italian Skillet Pasta**
  - Source recipes: Italian Skillet Dinner
  - Constraint notes: This dish satisfies the user's pasta main course constraint by providing a hearty and flavorful Italian skillet pasta dish. It includes penne pasta, ground beef, Italian sausage, and mozzarella cheese, which are all typical ingredients in Italian cuisine.
- **Dessert: Biscuit Tortoni**
  - Source recipes: Biscuit Tortoni(An Italian Dessert)
  - Constraint notes: This dish satisfies the user's dessert course constraint by providing a traditional Italian dessert. It includes evaporated milk, powdered sugar, and macaroon crumbs, which are all typical ingredients in Italian desserts.

**Overall notes:** The full menu fits together by providing a well-rounded and traditional Italian dining experience. Each course is carefully selected to showcase the flavors and ingredients of Italian cuisine, and the dishes are adapted to fit the user's specific requests.

Loaded demo_2: 24 retrieved recipes, 3 menu courses.


In [7]:
render_menu_demo("demo_3", "Demo 3: L3 - Low-fat Vegetarian")


### Demo 3: L3 - Low-fat Vegetarian

**Query:** Create a low-fat, vegetarian 3-course dinner (soup, main, dessert). Avoid using cheese or cream.

**Top-5 retrieved recipes:**
1. Vegetarian Stroganoff
2. Vegetarian Vegetable Soup
3. Surprise Fruit Salad
4. Chicken And Rice Dinner
5. Barley With Ham

**Generated menu:**
- **Starter: Vegetarian Vegetable Soup**
  - Source recipes: Vegetarian Vegetable Soup
  - Constraint notes: This dish satisfies the user's request for a low-fat, vegetarian starter by using a recipe that is inherently vegetarian and low in fat, avoiding cheese and cream. The 'Vegetarian Vegetable Soup' source recipe provides a clear foundation for this course.
- **Main: Vegetarian Stroganoff Adaptation**
  - Source recipes: Vegetarian Stroganoff
  - Constraint notes: To satisfy the user's request for a low-fat, vegetarian main course, we adapt the 'Vegetarian Stroganoff' recipe by removing the sour cream and Parmesan cheese, which are high in fat, and focusing on the vegetarian dinner steaks, mushrooms, and noodles. This adaptation ensures the dish remains vegetarian and reduces the fat content.
- **Dessert: Fresh Fruit Salad**
  - Source recipes: Surprise Fruit Salad
  - Constraint notes: For the dessert course, we draw inspiration from the 'Surprise Fruit Salad' but modify it to avoid the Cool Whip, which could be high in fat. Instead, we focus on fresh fruits like bananas, oranges, and other seasonal fruits, creating a light and refreshing dessert that is both vegetarian and low in fat.

**Overall notes:** This menu is designed to provide a balanced and satisfying vegetarian dining experience while adhering to the user's constraints of low fat and no cheese or cream. Each dish is carefully adapted from the source recipes to ensure compliance with these requirements, offering a delightful and healthy culinary journey.

Loaded demo_3: 24 retrieved recipes, 3 menu courses.


The three demos show that the pipeline can translate retrieval evidence into structured menus. L1 is the easiest case: the requested soup starter, chicken main, and cake dessert are direct course constraints, and the generated menu covers all of them. L2 requires a more coherent cuisine theme, and the pipeline uses Italian-style sources such as Antipasto Salad and Italian Skillet Pasta to support the starter and main course. L3 is the hardest because it combines vegetarian, low-fat, and avoidance constraints. The dessert choice is sensible because a fruit-based dish naturally supports a lighter menu. The LLM sometimes renames dishes, such as adapting “Surprise Fruit Salad” into “Fresh Fruit Salad”, but it preserves source attribution so the adaptation remains inspectable. This is useful for a menu-design task because exact copying is less important than grounded, constraint-aware adaptation.

## T3.3 LLM-as-Judge Evaluation

The evaluation uses two judge settings. The self-judge is Groq `llama-3.3-70b-versatile`, the same model family used for menu generation, while the cross-judge is Groq `llama-3.1-8b-instant`, a smaller model with different capability characteristics. This design intentionally exposes capability-dependent failure modes: a strong self-judge may be more fluent but more forgiving, while a smaller cross-judge may be harsher or less reliable. The judge prompt defines four dimensions: Constraint Satisfaction, Ingredient Faithfulness, Culinary Logic & Coherence, and Bias. It also includes a 1-5 rubric and explicitly asks for reasoning before score. The expected output is JSON, with each dimension containing a short reasoning field and an integer score, which makes downstream table construction straightforward. This structure also makes disagreements interpretable, because the numeric score can be checked against the written rationale rather than treated as an isolated judgment.

In [8]:
judge_md_path = ROOT / "outputs" / "judge_results.md"
judge_md = judge_md_path.read_text(encoding="utf-8")
start = judge_md.find("```text")
end = judge_md.find("```", start + len("```text"))
prompt_block = judge_md[start + len("```text"):end].strip() if start != -1 and end != -1 else judge_md

display(Markdown("```text\n" + prompt_block + "\n```"))
print("Displayed judge prompt template from outputs/judge_results.md.")


```text
Original user request:
{query}

Generated menu JSON:
{menu_json}

Retrieved source recipe evidence:
{sources_json}

Evaluate the menu on four dimensions:

1. Constraint Satisfaction
- Does the menu follow explicit user constraints such as number of courses, course types, cuisine, dietary restrictions, and avoided ingredients?

2. Ingredient Faithfulness
- Are dishes and claims grounded in the retrieved source recipes?
- Are source recipe titles cited accurately?
- Are adaptations reasonable and transparent?

3. Culinary Logic & Coherence
- Would the menu work as a coherent meal?
- Are course order, dish pairing, preparation logic, and flavor balance plausible?

4. Bias
- Does the menu avoid stereotypes, exclusionary assumptions, or culturally careless claims?
- Score high when the menu is neutral, respectful, and avoids unsupported cultural generalizations.

Rubric for each dimension:
1 = Poor: major failures or contradictions.
2 = Weak: several important issues.
3 = Adequate: mostly acceptable but with noticeable gaps.
4 = Good: satisfies the dimension with minor issues.
5 = Excellent: fully satisfies the dimension with clear evidence.

Important instructions:
- For each dimension, write the reasoning first, then choose the score.
- Scores must be integers from 1 to 5.
- Be critical but fair. Do not reward unsupported claims.
- Return this exact JSON shape:
{{
  "constraint_satisfaction": {{"reasoning": "...", "score": 1}},
  "ingredient_faithfulness": {{"reasoning": "...", "score": 1}},
  "culinary_logic_coherence": {{"reasoning": "...", "score": 1}},
  "bias": {{"reasoning": "...", "score": 1}}
}}
```

Displayed judge prompt template from outputs/judge_results.md.


In [9]:
with open(ROOT / "outputs" / "judge_results.json", "r", encoding="utf-8") as f:
    judge_results = json.load(f)

judge_rows = []
for item in judge_results:
    result = item["result"]
    metadata = result["metadata"]
    model_short = "70B" if "70b" in metadata["judge_model"].lower() else "8B"
    judge_rows.append({
        "Menu": item["demo"],
        "Eval": metadata["eval_type"],
        "Judge Model": model_short,
        "Constraint": result["constraint_satisfaction"]["score"],
        "Faithfulness": result["ingredient_faithfulness"]["score"],
        "Coherence": result["culinary_logic_coherence"]["score"],
        "Bias": result["bias"]["score"],
    })

judge_df = pd.DataFrame(judge_rows)
display(judge_df)
print(f"Loaded {len(judge_results)} judge result records from saved JSON.")


,Menu,Eval,Judge Model,Constraint,Faithfulness,Coherence,Bias
0,demo_1,self,70B,4,2,3,5
1,demo_1,cross,8B,2,4,3,4
2,demo_2,self,70B,5,2,4,5
3,demo_2,cross,8B,2,1,1,1
4,demo_3,self,70B,4,3,4,5
5,demo_3,cross,8B,2,3,3,4


Loaded 6 judge result records from saved JSON.


In [10]:
def find_judge(demo, eval_type):
    for item in judge_results:
        if item["demo"] == demo and item["result"]["metadata"]["eval_type"] == eval_type:
            return item
    raise KeyError((demo, eval_type))

success = find_judge("demo_3", "self")
failure = find_judge("demo_1", "cross")

reasoning_examples = pd.DataFrame([
    {
        "Example": "Successful self-judge reasoning",
        "Menu": "demo_3",
        "Eval": "self",
        "Dimension": "constraint_satisfaction",
        "Score": success["result"]["constraint_satisfaction"]["score"],
        "Reasoning": success["result"]["constraint_satisfaction"]["reasoning"],
    },
    {
        "Example": "Problematic cross-judge reasoning",
        "Menu": "demo_1",
        "Eval": "cross",
        "Dimension": "constraint_satisfaction",
        "Score": failure["result"]["constraint_satisfaction"]["score"],
        "Reasoning": failure["result"]["constraint_satisfaction"]["reasoning"],
    },
])

display(reasoning_examples)
print("Displayed one useful judge example and one inconsistent judge example for reflection.")


,Example,Menu,Eval,Dimension,Score,Reasoning
0,Successful self-judge reasoning,demo_3,self,constraint_satisfaction,4,The generated menu satisfies the user's reques...
1,Problematic cross-judge reasoning,demo_1,cross,constraint_satisfaction,2,The generated menu consists of 3 courses: a so...


Displayed one useful judge example and one inconsistent judge example for reflection.


## T3.4 Analysis and Reflection

The RAG pipeline was designed around reliability and interpretability rather than maximum retrieval complexity. The knowledge base uses all available recipes from train, development, and test, which is acceptable for Task 3 because no supervised model is trained from these documents. BM25 was chosen over dense retrieval because it is cheap to build, easy to run on CPU, deterministic, and strong for recipe queries where names and ingredients carry much of the meaning. The index text concatenates title, ingredients, and recipe instructions so that both high-level dish labels and detailed cooking evidence are available. Stopword filtering and plural normalization reduce false emphasis on generic request words and simple morphological variants. The generation stage improves ordinary top-k retrieval with multi-query enhancement: it retrieves a primary `top_k=12` context and then adds course-specific probes for soup, chicken, cake, Italian, pasta, vegetarian, and low-fat constraints. This is important because menu requests combine several course requirements that may not all appear in a single BM25 ranking. The LLM prompt enforces JSON output, exact source-title citation, and explicit constraint notes. The judge prompt mirrors this structure with reasoning-before-score and a clear 1-5 rubric, making the final evaluation auditable rather than only impressionistic. Overall, the design favors transparent evidence flow: query terms lead to recipe context, recipe context leads to menu items, and judge rationales explain how each menu item satisfies or fails the rubric.

The three menu levels reveal different strengths of the retrieval approach. L1 is mostly compositional: the query explicitly asks for soup, chicken, and cake, so keyword retrieval can find direct evidence for each course. The menu satisfies the requested structure and uses recognizable source titles. L2 adds a cuisine theme, making retrieval harder because “Italian” can appear in titles, ingredients, or not at all. The enhanced retrieval helps by adding probes for Italian pasta, Italian dessert, and antipasto-style starters, producing sources such as Antipasto Salad and Italian Skillet Pasta. L3 is the most difficult because the model must satisfy positive constraints, vegetarian and low-fat, plus negative constraints, avoiding cheese and cream. Retrieval alone cannot guarantee this because some retrieved vegetarian sources contain sour cream or Parmesan. The generation prompt therefore asks the LLM to adapt conservatively and explain avoidance in the constraint notes. This worked reasonably well for a fruit-based dessert and vegetable soup, although the source evidence sometimes required modification. Overall, BM25 is justified for the assignment because it is fast and transparent, but it remains limited by lexical overlap. A dense or hybrid retriever could improve cases where relevant dishes lack exact query terms. The current approach is therefore a pragmatic baseline: it is cheap enough for live demo use, explainable enough for inspection, and strong enough for common menu constraints.

The LLM-as-judge results are useful but clearly imperfect. One success case is the demo 3 self-judge constraint reasoning: it correctly recognizes that the generated menu addresses the vegetarian, low-fat, and no-cream constraints, while still noting that fat content is not numerically quantified. This is a thoughtful, calibrated judgment rather than a simple rubber stamp, because it gives credit for satisfying the explicit dietary request but preserves uncertainty about an unverifiable nutrition claim. The first failure mode is self-preference or sycophancy bias. The self-judge, using `llama-3.3-70b-versatile`, gave an average Constraint Satisfaction score of about 4.3, while the cross-judge, using `llama-3.1-8b-instant`, gave an average of 2.0. This systematic gap of more than two points suggests that a judge related to the generator may over-trust the menu's own explanations. The second failure mode is factual hallucination in smaller models. For demo 1, the cross-judge claimed that “Marinated Chicken Breasts does not match chicken main”, which is factually wrong because chicken breasts are plainly a chicken main dish. A third issue is source-versus-adaptation confusion. In demo 3, the cross-judge criticized cheese or cream even though the menu notes explicitly removed sour cream, Parmesan, and Cool Whip from adapted dishes. The judge appears to conflate the retrieved source recipe with the final adapted dish, so it penalizes evidence that was intentionally modified. Improvements should include stronger and multiple judge models, capability-aware ensembling, multi-sampling with majority voting, rubric calibration using anchor examples, and human-in-the-loop spot-checks for contradictory scores. The judge should also be independent from the generator model to improve provenance independence, and the prompt could separately ask judges to evaluate retrieved sources and final adapted dishes. Overall, LLM-as-judge is best treated as an analysis aid, not as a fully reliable ground truth evaluator.